# **Judicial Service Gen AI Chatbot For District Court**

---

## **Install Library**

---


In [1]:
pip install great_expectations pandas

Active code page: 1252
Note: you may need to restart the kernel to use updated packages.


## **Import Library**

---

In [2]:
import pandas as pd, json, pathlib
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
from great_expectations.checkpoint import SimpleCheckpoint

## **Connecting to a Datasource**

---

In [ ]:
# Load the combined service standards dataset generated from multiple source CSV files
df = pd.read_csv("Data/standar_layanan_combined.csv")

# Initialize the Great Expectations data context used for validations and data checks
context = gx.get_context()

In [ ]:
# Define a runtime pandas datasource using Great Expectations 0.18 style
DS_NAME = "local_pd"
RUNTIME_DC = "default_runtime_data_connector_name"
ASSET_NAME = "standar_layanan_df"

# Configuration for a simple pandas based datasource with a runtime data connector
datasource_config = {
    "name": DS_NAME,
    "class_name": "Datasource",
    "module_name": "great_expectations.datasource",
    "execution_engine": {
        "module_name": "great_expectations.execution_engine",
        "class_name": "PandasExecutionEngine",
    },
    "data_connectors": {
        RUNTIME_DC: {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"],
        }
    },
}

# Register the datasource only when it is not already present in the context
try:
    context.get_datasource(DS_NAME)
except Exception:
    context.add_datasource(**datasource_config)

# Build a RuntimeBatchRequest that wraps the in memory DataFrame
batch_request = RuntimeBatchRequest(
    datasource_name=DS_NAME,
    data_connector_name=RUNTIME_DC,
    data_asset_name=ASSET_NAME,
    runtime_parameters={"batch_data": df},  # in memory DataFrame
    batch_identifiers={"default_identifier_name": "default_id"},
)

# Create or update the expectation suite and then build a validator for this batch
SUITE_NAME = "standar_layanan_suite_auto"
context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=SUITE_NAME,
)


In [5]:
# Batch preview
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,category,question,answer
0,STANDAR PELAYANAN PENDAFTARAN SURAT KETERANGAN...,Dalam ketentuan layanan standar pelayanan pend...,Dokumen yang harus disiapkan mencakup surat pe...
1,STANDAR PELAYANAN PENDAFTARAN SURAT KETERANGAN...,Dalam ketentuan layanan standar pelayanan pend...,Dokumen yang harus disiapkan mencakup surat pe...
2,STANDAR PELAYANAN PENDAFTARAN SURAT KETERANGAN...,Dalam ketentuan layanan standar pelayanan pend...,Dokumen yang harus disiapkan mencakup surat pe...
3,STANDAR PELAYANAN PENDAFTARAN SURAT KETERANGAN...,Dalam ketentuan layanan standar pelayanan pend...,Dokumen yang harus disiapkan mencakup surat pe...
4,STANDAR PELAYANAN PENDAFTARAN SURAT KETERANGAN...,Dalam ketentuan layanan standar pelayanan pend...,Dokumen yang harus disiapkan mencakup surat pe...


In [6]:
# Batch columns
validator.columns()

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

['category', 'question', 'answer']

In [ ]:
# Expectations using the method based API in Great Expectations 0.18.x
STRICT_CATEGORY_LIST = False
CATEGORY_MOSTLY = 0.99

# Table level checks for basic shape of the dataset
validator.expect_table_row_count_to_be_between(min_value=1)
validator.expect_table_column_count_to_equal(value=3)
validator.expect_table_columns_to_match_set(column_set=["category", "question", "answer"])

# Cleanliness checks for key text columns
for col in ["category", "question", "answer"]:
    validator.expect_column_values_to_not_be_null(column=col)
    validator.expect_column_values_to_not_match_regex(column=col, regex=r"\s{2,}")   # prevent double spaces
    validator.expect_column_values_to_not_match_regex(column=col, regex=r"<[^>]+>")  # block HTML fragments
    validator.expect_column_values_to_not_match_regex(column=col, regex=r"^\s")      # block leading spaces
    validator.expect_column_values_to_not_match_regex(column=col, regex=r"\s$")      # block trailing spaces

# Category column expectations
validator.expect_column_value_lengths_to_be_between(column="category", min_value=1, max_value=140)
validator.expect_column_unique_value_count_to_be_between(
    column="category",
    min_value=30,
    max_value=60,
)
if STRICT_CATEGORY_LIST:
    allowed = sorted(df["category"].dropna().astype(str).unique().tolist())
    validator.expect_column_values_to_be_in_set(
        column="category", value_set=allowed, mostly=CATEGORY_MOSTLY
    )

# Question column expectations
validator.expect_column_value_lengths_to_be_between(column="question", min_value=40, max_value=180)
validator.expect_column_values_to_match_regex(column="question", regex=r".+\?$", mostly=0.90)
validator.expect_column_values_to_match_regex(column="question", regex=r"^[A-Z0-9À-Ý]")
validator.expect_column_proportion_of_unique_values_to_be_between(
    column="question",
    min_value=0.88,
    max_value=1,
)

# Answer column expectations
validator.expect_column_value_lengths_to_be_between(column="answer", min_value=150, max_value=2000)
validator.expect_column_values_to_match_regex(column="answer", regex=r"^[A-Z0-9À-Ý]")
validator.expect_column_values_to_not_match_regex(column="answer", regex=r"https?://", mostly=0.99)

# Duplicate row checks across key business keys
validator.expect_compound_columns_to_be_unique(column_list=["category", "question", "answer"])
validator.expect_compound_columns_to_be_unique(column_list=["category", "question"], mostly=0.998)

# Persist the expectation suite definition for reuse
validator.save_expectation_suite(discard_failed_expectations=False)

# Step 1 Build a dict style batch_request required inside a checkpoint
batch_request_dict = {
    "datasource_name": DS_NAME,
    "data_connector_name": RUNTIME_DC,
    "data_asset_name": ASSET_NAME,
    "runtime_parameters": {"batch_data": df},              # in memory DataFrame
    "batch_identifiers": {"default_identifier_name": "run"},
}

# Step 2 Create a checkpoint without static validations and pass validations at run time
CHECKPOINT_NAME = "val_auto_standar_layanan_all"
checkpoint = SimpleCheckpoint(
    name=CHECKPOINT_NAME,
    data_context=context,
)

# Run the checkpoint for the given batch and expectation suite
result = checkpoint.run(validations=[{
    "batch_request": batch_request_dict,
    "expectation_suite_name": SUITE_NAME,
}])

# Step 3 Print a detailed view of every expectation that was evaluated
rows = []
all_json = []

for run_key, payload in result.run_results.items():
    vr = payload["validation_result"]   # ExpectationSuiteValidationResult
    all_json.append(vr.to_json_dict())

    # High level summary for this validation run
    print(f"\n==== Validation: {run_key} | success={vr.success} ====")
    print(vr.statistics)

    # Row wise summary for each individual expectation result
    for er in vr.results:
        cfg = er.expectation_config
        res = er.result or {}

        sample_unexp = (
            res.get("partial_unexpected_list")
            or res.get("unexpected_list")
            or []
        )[:10]  # sample of values that violated the expectation

        rows.append({
            "expectation": cfg.expectation_type,
            "column": cfg.kwargs.get("column"),
            "success": er.success,
            "mostly": cfg.kwargs.get("mostly"),
            "unexpected_count": res.get("unexpected_count"),
            "unexpected_percent": res.get("unexpected_percent"),
            "observed_value": res.get("observed_value"),
            "sample_unexpected": sample_unexp,
            # keep remaining kwargs so important context is not lost
            "kwargs": {k: v for k, v in cfg.kwargs.items() if k not in ("column", "mostly")},
        })

# Build a compact table summarizing all expectations
df_results = pd.DataFrame(rows)
if not df_results.empty:
    ordered_cols = [
        "expectation", "column", "success", "mostly",
        "unexpected_count", "unexpected_percent", "observed_value",
        "sample_unexpected", "kwargs",
    ]
    df_results = df_results.reindex(columns=[c for c in ordered_cols if c in df_results.columns])
    print("\n=== Semua expectation (ringkas) ===")
    print(df_results.to_string(index=False, max_colwidth=80))
else:
    print("\nTidak ada hasil expectation di-run.")

# Optional persist validation outputs to disk for offline review
df_results.to_csv("validation_results_summary.csv", index=False, encoding="utf-8")
Path("validation_results_full.json").write_text(
    json.dumps(all_json, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print('\nFile tersimpan: "validation_results_summary.csv" dan "validation_results_full.json"')


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/3 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/77 [00:00<?, ?it/s]


==== Validation: ValidationResultIdentifier::standar_layanan_suite_auto/__none__/20251116T141930.004394Z/a9bcd9603f1cf3610daf62b791404701 | success=False ====
{'evaluated_expectations': 18, 'successful_expectations': 14, 'unsuccessful_expectations': 4, 'success_percent': 77.77777777777779}

=== Semua expectation (ringkas) ===
                                            expectation   column  success  mostly  unexpected_count  unexpected_percent               observed_value                                                                sample_unexpected                                                                           kwargs
                   expect_table_row_count_to_be_between     None     True     NaN               NaN                 NaN                         6000                                                                               []                 {'min_value': 1, 'batch_id': 'a9bcd9603f1cf3610daf62b791404701'}
                     expect_table_column_count_to

**Insight:**

The FAQ dataset contains 6,000 rows and three columns (category, question, answer) with 47 unique categories; schema checks and basic cleanliness (non-nulls, no trailing spaces, virtually no URLs) pass, and of 18 expectations evaluated, 14 passed and 4 failed (77.78% success), including the corrected proportion-based uniqueness for question which passes at 0.897 ≥ 0.88. Flags on compound uniqueness—about 13.75% for (category,question,answer) and for “almost unique” (category,question)—are not a concern here because, after review, there is no truly identical question pattern; the repeats reflect natural topical clustering. Likewise, ~7.63% length violations for question are acceptable since the seemingly verbose phrasing is intentionally retained as a strong hint for categorization, and ~2.58% short answer entries are also acceptable because those responses are meant to be concise by design. Under these curatorial choices, the corpus is coherent and suitable for your intended use.


---